In [ ]:
import re
import time
import pandas as pd
from tqdm.notebook import tqdm
from openai import OpenAI
from pathlib import Path

In [ ]:
INPUT_FILE = Path("../data/goemotions.csv")
OUTPUT_DIR = Path("../results_prompts_llm")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FILE = OUTPUT_DIR / "llm_emotion_results.csv"

print(f"Input : {INPUT_FILE}")
print(f"Output: {OUTPUT_FILE}")

In [ ]:
INPUT_FILES = [Path("../data/goemotions.csv")]
COLS_TO_DROP = [
    "id", "example_very_unclear", "author", "subreddit",
    "link_id", "parent_id", "created_utc", "rater_id",
]
MODELS = {
    "gpt-4o":           "openai/gpt-4o",
    "claude-sonnet":    "anthropic/claude-sonnet-4.5",
    "gemini-2.5-pro":   "google/gemini-2.5-pro",
    "grok-3":           "x-ai/grok-3",
}
SLEEP_BETWEEN_CALLS = 4  
MAX_ROWS = 20

In [ ]:
OPENROUTER_API_KEY = Path("../secrets/openrouter_key.txt").read_text().strip()
client = OpenAI(
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1",
)

In [10]:
def load_and_prepare_data(input_files, cols_to_drop):
    frames = []
    for f in input_files:
        if Path(f).exists():
            frames.append(pd.read_csv(f, sep=",", header=0))
        else:
            print(f"File not found, skipping: {f}")
    if not frames:
        raise FileNotFoundError("No input files could be loaded.")
    df = pd.concat(frames, ignore_index=True)
    existing_drop = [c for c in cols_to_drop if c in df.columns]
    df = df.drop(columns=existing_drop).drop_duplicates()
    emotion_columns = [c for c in df.columns if c != "text"]
    df["emotions"] = df[emotion_columns].apply(
        lambda row: ", ".join(row.index[row == 1]), axis=1
    )
    df = df[["text", "emotions"]].reset_index(drop=True)
    print(f"Loaded {len(df):,} unique sentences across {len(emotion_columns)} emotion classes.")
    print(f"Emotions: {emotion_columns}")
    return df, emotion_columns

In [11]:
SYSTEM_PROMPT = (
    "You are an expert in emotion analysis. "
    "When given a sentence, you identify which emotions from a provided list "
    "are expressed in it. You always end your answer with a line that starts with "
    "'Emotions:' followed by a comma-separated list of the emotions you detected "
    "(use only names from the provided list, nothing else on that line)."
)

In [12]:
def build_user_prompt(sentence, emotion_list):
    emotions_str = ", ".join(emotion_list)
    return (
        f'Sentence: "{sentence}"\n\n'
        f"Choose one or more emotions expressed in the sentence from this list:\n"
        f"{emotions_str}\n\n"
        f"Explain your reasoning briefly, then end with:\n"
        f"Emotions: <comma-separated emotions from the list above>"
    )

In [13]:
def extract_emotions(reply, valid_emotions):
    match = re.search(r"(?i)emotions?\s*:\s*(.+)", reply)
    if match:
        candidates = [e.strip().lower() for e in match.group(1).split(",")]
        found = [e for e in valid_emotions if e.lower() in candidates]
        if found:
            return ", ".join(found)
    reply_lower = reply.lower()
    found = [e for e in valid_emotions if re.search(rf"\b{re.escape(e.lower())}\b", reply_lower)]
    return ", ".join(found) if found else "none_found"


In [14]:
def query_model(model_id, sentence, emotion_list, retries=3):
    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model=model_id,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": build_user_prompt(sentence, emotion_list)},
                ],
                temperature=0.2,
                max_tokens=2000,
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            wait = 2 ** attempt
            print(f"Attempt {attempt+1}/{retries} failed ({e}). Retrying in {wait}s...")
            time.sleep(wait)
    return "ERROR: all retries failed"

print("All helpers defined.")

All helpers defined.


In [ ]:
df, emotion_columns = load_and_prepare_data(INPUT_FILES, COLS_TO_DROP)
if MAX_ROWS:
    df = df.head(MAX_ROWS)
    print(f"MAX_ROWS={MAX_ROWS} - using a subset for testing.")



Loaded 154,135 unique sentences across 28 emotion classes.
Emotions: ['admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval', 'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief', 'joy', 'love', 'nervousness', 'optimism', 'pride', 'realization', 'relief', 'remorse', 'sadness', 'surprise', 'neutral']
MAX_ROWS=20 - using a subset for testing.


In [16]:
df = df.drop(index=1)
df

,text,emotions
0,That game hurt.,sadness
2,"You do right, if you don't care then fuck 'em!",neutral
3,Man I love reddit.,love
4,"[NAME] was nowhere near them, he was by the Fa...",neutral
5,Right? Considering it’s such an important docu...,gratitude
6,"He isn't as big, but he's still quite popular....",disapproval
7,That's crazy; I went to a super [RELIGION] hig...,amusement
8,that's adorable asf,amusement
9,"""Sponge Blurb Pubs Quaw Haha GURR ha AAa!"" fin...",amusement
10,"I have, and now that you mention it, I think t...",neutral


In [17]:
### Run the pipeline

if OUTPUT_FILE.exists():
    existing = pd.read_csv(OUTPUT_FILE)
    print(f"Resuming: {len(existing):,} rows already saved.")
else:
    existing = pd.DataFrame(columns=["model", "sentence", "answer_from_the_model", "emotion_found"])
    print("Starting fresh.")

new_rows = []

for model_label, model_id in MODELS.items():
    print(f"\n{'='*65}")
    print(f"Model: {model_label}  ({model_id})")
    print(f"{'='*65}")

    already_done = set(
        existing.loc[existing["model"] == model_label, "sentence"]
    ) if not existing.empty else set()

    sentences_to_run = df[~df["text"].isin(already_done)]
    print(f"{len(sentences_to_run):,} to process - {len(already_done):,} already done.\n")

    for _, row in tqdm(sentences_to_run.iterrows(), total=len(sentences_to_run), desc=model_label):
        sentence = row["text"]

        reply         = query_model(model_id, sentence, emotion_columns)
        emotion_found = extract_emotions(reply, emotion_columns)

        new_rows.append({
            "model":                 model_label,
            "sentence":              sentence,
            "answer_from_the_model": reply,
            "emotion_found":         emotion_found,
            "true_emotions":         row["emotions"],
        })

        print(f"\n{'─'*60}")
        print(f"Model    : {model_label}")
        print(f"Sentence : {sentence[:120]}{'...' if len(sentence) > 120 else ''}")
        print(f"Reply    : {reply[:300]}{'...' if len(reply) > 300 else ''}")
        print(f"Emotions : {emotion_found}")
        print(f"True     : {row['emotions']}")

        time.sleep(SLEEP_BETWEEN_CALLS)

    if new_rows:
        combined = pd.concat([existing, pd.DataFrame(new_rows)], ignore_index=True)
        combined.to_csv(OUTPUT_FILE, index=False)
        print(f"\nSaved {len(combined):,} total rows to {OUTPUT_FILE}")

print(f"\nPipeline complete! Results at: {OUTPUT_FILE}")

Starting fresh.

Model: gpt-4o  (openai/gpt-4o)
19 to process - 0 already done.



gpt-4o:   0%|          | 0/19 [00:00<?, ?it/s]


────────────────────────────────────────────────────────────
Model    : gpt-4o
Sentence : That game hurt.
Reply    : The sentence "That game hurt" suggests an emotional response to an event, likely indicating a negative experience. The word "hurt" implies emotional pain or discomfort, which aligns with feelings of disappointment or sadness. The context of a game suggests that the speaker might have had expectation...
Emotions : disappointment, sadness
True     : sadness

────────────────────────────────────────────────────────────
Model    : gpt-4o
Sentence : You do right, if you don't care then fuck 'em!
Reply    : The sentence expresses a strong sense of disapproval and possibly anger towards others, as indicated by the use of the phrase "fuck 'em." There is also a sense of approval or pride in the decision to not care about what others think, as suggested by the phrase "You do right." The language used conve...
Emotions : anger, disapproval, pride
True     : neutral

──────────────

claude-sonnet:   0%|          | 0/19 [00:00<?, ?it/s]


────────────────────────────────────────────────────────────
Model    : claude-sonnet
Sentence : That game hurt.
Reply    : The sentence "That game hurt" expresses emotional pain or distress about a game's outcome. Let me analyze the possible emotions:

**Reasoning:**
- The word "hurt" indicates emotional pain, suggesting negative feelings about the game
- This could reflect **disappointment** - the speaker is let down b...
Emotions : disappointment, grief, sadness
True     : sadness

────────────────────────────────────────────────────────────
Model    : claude-sonnet
Sentence : You do right, if you don't care then fuck 'em!
Reply    : Let me analyze the emotions in this sentence.

The sentence "You do right, if you don't care then fuck 'em!" contains:

1. **Approval** - "You do right" is explicitly approving of someone's action or attitude
2. **Disapproval** - The phrase "fuck 'em" expresses strong disapproval or dismissal of oth...
Emotions : anger, approval, disapproval
True     :

gemini-2.5-pro:   0%|          | 0/19 [00:00<?, ?it/s]


────────────────────────────────────────────────────────────
Model    : gemini-2.5-pro
Sentence : That game hurt.
Reply    : The word "hurt" in this context refers to emotional pain resulting from the outcome of the game. This strongly suggests the speaker is experiencing `disappointment` because the result was not what they had hoped for. This feeling of being let down and the emotional pain it caused can also be describ...
Emotions : anger, disappointment, sadness
True     : sadness

────────────────────────────────────────────────────────────
Model    : gemini-2.5-pro
Sentence : You do right, if you don't care then fuck 'em!
Reply    : This sentence expresses a complex mix of emotions directed at different parties.

The phrase "You do right" is a clear statement of validation and support for the person being addressed, which signifies **Approval**.

The second part, "...if you don't care then fuck 'em!", is an aggressive and conte...
Emotions : anger, approval, disapproval, disgust

grok-3:   0%|          | 0/19 [00:00<?, ?it/s]


────────────────────────────────────────────────────────────
Model    : grok-3
Sentence : That game hurt.
Reply    : The sentence "That game hurt" suggests a negative emotional experience related to the game, likely indicating emotional pain or frustration. The word "hurt" typically conveys a sense of sadness or disappointment, as it implies being emotionally affected in a negative way, possibly due to a loss or a...
Emotions : disappointment, sadness
True     : sadness

────────────────────────────────────────────────────────────
Model    : grok-3
Sentence : You do right, if you don't care then fuck 'em!
Reply    : The sentence "You do right, if you don't care then fuck 'em!" conveys a strong, dismissive attitude toward others' opinions, combined with a sense of defiance or frustration. The use of the phrase "fuck 'em" suggests irritation or anger toward those who might oppose or judge the speaker's actions. A...
Emotions : anger, approval
True     : neutral

────────────────────────

In [18]:
### Check the result
results = pd.read_csv(OUTPUT_FILE)
print(f"Total rows: {len(results):,}")
print(f"\nRows per model:")
print(results["model"].value_counts().to_string())
print(f"\nMost common predicted emotions:")
print(results["emotion_found"].value_counts().head(15).to_string())
results.head(10)

Total rows: 76

Rows per model:
model
gpt-4o            19
claude-sonnet     19
gemini-2.5-pro    19
grok-3            19

Most common predicted emotions:
emotion_found
neutral                                  11
realization, surprise                     3
amusement                                 3
approval, neutral                         3
amusement, optimism                       2
disappointment, sadness                   2
joy, love                                 2
amusement, realization, surprise          2
annoyance, disapproval, disgust, fear     2
amusement, annoyance, disapproval         2
annoyance, gratitude                      2
approval, desire, gratitude, optimism     2
amusement, approval, optimism             2
caring, disappointment, disapproval       1
approval, gratitude, optimism             1


,model,sentence,answer_from_the_model,emotion_found,true_emotions
0,gpt-4o,That game hurt.,"The sentence ""That game hurt"" suggests an emot...","disappointment, sadness",sadness
1,gpt-4o,"You do right, if you don't care then fuck 'em!",The sentence expresses a strong sense of disap...,"anger, disapproval, pride",neutral
2,gpt-4o,Man I love reddit.,"The sentence ""Man I love reddit."" expresses a ...","joy, love",love
3,gpt-4o,"[NAME] was nowhere near them, he was by the Fa...","The sentence ""[NAME] was nowhere near them, he...",neutral,neutral
4,gpt-4o,Right? Considering it’s such an important docu...,"The sentence expresses a sense of annoyance, a...","annoyance, gratitude",gratitude
5,gpt-4o,"He isn't as big, but he's still quite popular....",The sentence expresses a neutral observation a...,neutral,disapproval
6,gpt-4o,That's crazy; I went to a super [RELIGION] hig...,"The sentence expresses a sense of surprise, as...","realization, surprise",amusement
7,gpt-4o,that's adorable asf,"The sentence ""that's adorable asf"" expresses a...","joy, love",amusement
8,gpt-4o,"""Sponge Blurb Pubs Quaw Haha GURR ha AAa!"" fin...","The sentence refers to a finale being ""too rea...","amusement, realization, surprise",amusement
9,gpt-4o,"I have, and now that you mention it, I think t...","The sentence expresses a sense of nostalgia, w...",realization,neutral
